# 10 — Siamese CNN V3 + Class Proxy Regularization

## Obiettivo

Mantenere la pipeline Siamese basata su Euclidean Contrastive Learning,
aggiungendo un vincolo class-aware per organizzare meglio lo spazio
degli embedding rispetto alle 12 classi Tumor + Healthy.

La Contrastive Loss continua a ottimizzare la similarità tra coppie:

- stessa classe → embedding vicini
- classi diverse → embedding distanti

In aggiunta, ogni classe possiede un proxy apprendibile nello spazio
latente. Ogni embedding viene incoraggiato ad avvicinarsi al proxy
della propria classe.

Loss totale:

L = L_contrastive + lambda_proxy * L_proxy

Configurazione iniziale:

- FCGR k=6, 64×64
- CNN Encoder V3
- embedding 128D L2-normalizzato
- Euclidean Contrastive Loss
- margin = 1.25
- random pairs 50/50
- 12 classi Tumor + Healthy
- downsampling a 2111 campioni/classe
- lambda_proxy = 0.10

In [1]:
# ============================================================
# IMPORTS + CONFIG
# ============================================================

from pathlib import Path

import random
import time
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)


# ============================================================
# PATHS
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "siamese_contrastive_proxy_k6"
)

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_manifest.tsv"
)

CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_class_mapping.tsv"
)

VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)


# ============================================================
# FCGR k=6
# ============================================================

K = 6
FCGR_SIZE = 64

FCGR_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / "fcgr_k6.npy"
)

FCGR_INDEX_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / "fcgr_k6_index.tsv"
)


# ============================================================
# EXPERIMENT
# ============================================================

RANDOM_STATE = 42

N_CLASSES = 12

EMBEDDING_DIM = 128

EUCLIDEAN_MARGIN = 1.25

TRAIN_PAIRS_PER_EPOCH = 50_000

VAL_PAIRS = 10_000

POSITIVE_FRACTION = 0.50

BATCH_SIZE = 64

LEARNING_RATE = 3e-4

WEIGHT_DECAY = 1e-4


# ============================================================
# PROXY CONFIG
# ============================================================

LAMBDA_PROXY = 0.10

PROXY_TEMPERATURE = 0.10


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AMP_ENABLED = DEVICE.type == "cuda"


print("=" * 72)
print("SIAMESE CONTRASTIVE + PROXY")
print("=" * 72)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print()
print("FCGR k:", K)
print("FCGR size:", FCGR_SIZE)
print("Embedding:", EMBEDDING_DIM)
print("Margin:", EUCLIDEAN_MARGIN)
print("Lambda proxy:", LAMBDA_PROXY)
print("Proxy temperature:", PROXY_TEMPERATURE)

SIAMESE CONTRASTIVE + PROXY
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU

FCGR k: 6
FCGR size: 64
Embedding: 128
Margin: 1.25
Lambda proxy: 0.1
Proxy temperature: 0.1


In [2]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_STATE)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision(
        "high"
    )

In [3]:
# ============================================================
# LOAD MANIFEST
# ============================================================

metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={"id": str}
)

metadata["class_id"] = (
    metadata["class_id"]
    .astype(int)
)

class_mapping = pd.read_csv(
    CLASS_MAPPING_PATH,
    sep="\t"
)


# ============================================================
# TRAIN
# ============================================================

full_train_metadata = (
    metadata[
        metadata["split_cluster"]
        ==
        "train"
    ]
    .copy()
    .reset_index(drop=True)
)


train_counts = (
    full_train_metadata["class_id"]
    .value_counts()
    .sort_index()
)


MIN_CLASS_SIZE = int(
    train_counts.min()
)


balanced_parts = []


for class_id in sorted(
    full_train_metadata["class_id"]
    .unique()
):

    class_df = (
        full_train_metadata[
            full_train_metadata["class_id"]
            ==
            class_id
        ]
    )

    sampled = class_df.sample(
        n=MIN_CLASS_SIZE,
        replace=False,
        random_state=(
            RANDOM_STATE
            +
            int(class_id)
        )
    )

    balanced_parts.append(
        sampled
    )


train_metadata = (
    pd.concat(
        balanced_parts,
        ignore_index=True
    )
    .reset_index(drop=True)
)


print("=" * 72)
print("TRAIN")
print("=" * 72)

print(
    "Full train:",
    len(full_train_metadata)
)

print(
    "Samples/class:",
    MIN_CLASS_SIZE
)

print(
    "Balanced:",
    len(train_metadata)
)

print(
    "Classes:",
    train_metadata["class_id"]
    .nunique()
)

TRAIN
Full train: 96167
Samples/class: 2111
Balanced: 25332
Classes: 12


In [6]:
# ============================================================
# VALIDATION
# ============================================================

old_to_new = dict(
    zip(
        class_mapping[
            "original_class_id"
        ].astype(int),

        class_mapping[
            "class_id"
        ].astype(int)
    )
)


included_original_ids = set(
    old_to_new.keys()
)


val_original = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
    dtype={"id": str}
)


val_original["class_id"] = (
    val_original["class_id"]
    .astype(int)
)


val_metadata = (
    val_original[
        val_original["class_id"]
        .isin(
            included_original_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


val_metadata[
    "original_class_id"
] = val_metadata[
    "class_id"
]


val_metadata[
    "class_id"
] = (
    val_metadata[
        "original_class_id"
    ]
    .map(old_to_new)
    .astype(int)
)


print(
    "Validation samples:",
    len(val_metadata)
)

print(
    "Validation classes:",
    val_metadata["class_id"]
    .nunique()
)

Validation samples: 9753
Validation classes: 12


In [7]:
# ============================================================
# FCGR k=6
# ============================================================

fcgr_memmap = np.load(
    FCGR_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={"id": str}
)


id_to_fcgr_row = dict(
    zip(
        fcgr_index["id"],
        fcgr_index["fcgr_row"]
    )
)


missing_train = (
    ~train_metadata["id"]
    .isin(id_to_fcgr_row)
).sum()


missing_val = (
    ~val_metadata["id"]
    .isin(id_to_fcgr_row)
).sum()


print("=" * 72)
print("FCGR k=6")
print("=" * 72)

print(
    "Shape:",
    fcgr_memmap.shape
)

print(
    "Missing train:",
    missing_train
)

print(
    "Missing validation:",
    missing_val
)


assert fcgr_memmap.shape[1:] == (
    64,
    64
)

assert missing_train == 0
assert missing_val == 0

FCGR k=6
Shape: (150272, 64, 64)
Missing train: 0
Missing validation: 0


In [8]:
# ============================================================
# RANDOM PAIRS + CLASS LABELS
# ============================================================

class RandomProxyPairDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row,
        n_pairs,
        positive_fraction=0.50,
        seed=42,
        dynamic=True
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = fcgr_memmap

        self.n_pairs = int(n_pairs)

        self.positive_fraction = float(
            positive_fraction
        )

        self.seed = int(seed)

        self.dynamic = bool(dynamic)


        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )


        self.labels = (
            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )


        self.classes = np.array(
            sorted(
                np.unique(
                    self.labels
                )
            ),
            dtype=np.int64
        )


        self.class_to_indices = {

            int(c):
                np.where(
                    self.labels == c
                )[0]

            for c in self.classes
        }


        self._generate_pairs(
            self.seed
        )


    def _generate_pairs(
        self,
        seed
    ):

        rng = np.random.default_rng(
            seed
        )


        n_positive = int(
            round(
                self.n_pairs
                *
                self.positive_fraction
            )
        )


        targets = np.zeros(
            self.n_pairs,
            dtype=np.float32
        )

        targets[:n_positive] = 1.0

        rng.shuffle(targets)


        idx1 = rng.integers(
            0,
            len(self.labels),
            size=self.n_pairs
        )


        idx2 = np.empty(
            self.n_pairs,
            dtype=np.int64
        )


        for i in range(
            self.n_pairs
        ):

            anchor_idx = int(
                idx1[i]
            )

            anchor_class = int(
                self.labels[
                    anchor_idx
                ]
            )


            if targets[i] == 1.0:

                candidates = (
                    self.class_to_indices[
                        anchor_class
                    ]
                )

                partner_idx = anchor_idx

                while (
                    partner_idx
                    ==
                    anchor_idx
                ):

                    partner_idx = int(
                        rng.choice(
                            candidates
                        )
                    )


            else:

                negative_classes = (
                    self.classes[
                        self.classes
                        !=
                        anchor_class
                    ]
                )


                negative_class = int(
                    rng.choice(
                        negative_classes
                    )
                )


                partner_idx = int(
                    rng.choice(
                        self.class_to_indices[
                            negative_class
                        ]
                    )
                )


            idx2[i] = partner_idx


        self.row1 = self.rows[idx1]

        self.row2 = self.rows[idx2]

        self.class1 = self.labels[idx1]

        self.class2 = self.labels[idx2]

        self.targets = targets


    def set_epoch(
        self,
        epoch
    ):

        if self.dynamic:

            self._generate_pairs(
                self.seed
                +
                int(epoch)
                *
                100_003
            )


    def __len__(self):

        return self.n_pairs


    def __getitem__(
        self,
        index
    ):

        x1 = np.array(
            self.fcgr_memmap[
                int(
                    self.row1[index]
                )
            ],
            dtype=np.float32,
            copy=True
        )


        x2 = np.array(
            self.fcgr_memmap[
                int(
                    self.row2[index]
                )
            ],
            dtype=np.float32,
            copy=True
        )


        return {

            "x1":
                torch.from_numpy(
                    x1
                ).unsqueeze(0),

            "x2":
                torch.from_numpy(
                    x2
                ).unsqueeze(0),

            "target":
                torch.tensor(
                    self.targets[index],
                    dtype=torch.float32
                ),

            "class1":
                torch.tensor(
                    self.class1[index],
                    dtype=torch.long
                ),

            "class2":
                torch.tensor(
                    self.class2[index],
                    dtype=torch.long
                )
        }

In [9]:
# ============================================================
# PAIR LOADERS
# ============================================================

train_pair_dataset = RandomProxyPairDataset(
    metadata=train_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row,
    n_pairs=TRAIN_PAIRS_PER_EPOCH,
    positive_fraction=0.50,
    seed=RANDOM_STATE,
    dynamic=True
)


val_pair_dataset = RandomProxyPairDataset(
    metadata=val_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row,
    n_pairs=VAL_PAIRS,
    positive_fraction=0.50,
    seed=RANDOM_STATE + 50_000,
    dynamic=False
)


train_pair_loader = DataLoader(
    train_pair_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=AMP_ENABLED
)


val_pair_loader = DataLoader(
    val_pair_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=AMP_ENABLED
)


print(
    "Train pairs:",
    len(train_pair_dataset)
)

print(
    "Positive fraction:",
    train_pair_dataset.targets.mean()
)

print(
    "Validation pairs:",
    len(val_pair_dataset)
)

Train pairs: 50000
Positive fraction: 0.5
Validation pairs: 10000


In [10]:
# ============================================================
# CNN V3
# ============================================================

class FCGRCNNEncoderV3(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()


        self.features = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                32
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv2d(
                32,
                32,
                3,
                padding=1,
                bias=False
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                32,
                64,
                3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                64,
                128,
                3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                128,
                128,
                3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                128
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.AdaptiveAvgPool2d(
                (4, 4)
            )
        )


        self.embedding_head = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Linear(
                256,
                embedding_dim
            )
        )


    def forward(
        self,
        x
    ):

        x = self.features(x)

        z = self.embedding_head(x)

        return F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8
        )

In [11]:
# ============================================================
# SIAMESE + CLASS PROXIES
# ============================================================

class SiameseWithProxies(nn.Module):

    def __init__(
        self,
        n_classes=12,
        embedding_dim=128,
        temperature=0.10
    ):

        super().__init__()


        self.encoder = FCGRCNNEncoderV3(
            embedding_dim=embedding_dim
        )


        self.proxies = nn.Parameter(
            torch.randn(
                n_classes,
                embedding_dim
            )
        )


        nn.init.normal_(
            self.proxies,
            mean=0.0,
            std=0.02
        )


        self.temperature = float(
            temperature
        )


    def forward(
        self,
        x1,
        x2
    ):

        batch_size = x1.shape[0]


        x = torch.cat(
            [x1, x2],
            dim=0
        )


        z = self.encoder(x)


        return (
            z[:batch_size],
            z[batch_size:]
        )


    def proxy_logits(
        self,
        z
    ):

        normalized_proxies = F.normalize(
            self.proxies,
            p=2,
            dim=1
        )


        cosine_similarity = (
            z
            @
            normalized_proxies.T
        )


        return (
            cosine_similarity
            /
            self.temperature
        )


model = SiameseWithProxies(
    n_classes=N_CLASSES,
    embedding_dim=EMBEDDING_DIM,
    temperature=PROXY_TEMPERATURE
).to(DEVICE)


print("=" * 72)
print("SIAMESE + PROXIES")
print("=" * 72)

print(
    "Proxy shape:",
    model.proxies.shape
)

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

SIAMESE + PROXIES
Proxy shape: torch.Size([12, 128])
Trainable parameters: 808,800


In [12]:
# ============================================================
# CONTRASTIVE LOSS
# ============================================================

class EuclideanContrastiveLoss(nn.Module):

    def __init__(
        self,
        margin=1.25
    ):

        super().__init__()

        self.margin = float(
            margin
        )


    def forward(
        self,
        z1,
        z2,
        target
    ):

        distances = torch.linalg.vector_norm(
            z1.float()
            -
            z2.float(),
            ord=2,
            dim=1
        )


        positive_loss = (
            target
            *
            distances.pow(2)
        )


        negative_loss = (
            (1.0 - target)
            *
            F.relu(
                self.margin
                -
                distances
            ).pow(2)
        )


        return (
            (
                positive_loss
                +
                negative_loss
            ).mean(),
            distances
        )


contrastive_criterion = (
    EuclideanContrastiveLoss(
        margin=EUCLIDEAN_MARGIN
    )
)


proxy_criterion = (
    nn.CrossEntropyLoss()
)

In [13]:
# ============================================================
# COMBINED SIAMESE + PROXY LOSS
# ============================================================

def compute_combined_loss(
    model,
    z1,
    z2,
    target,
    class1,
    class2,
    lambda_proxy
):

    contrastive_loss, distances = (
        contrastive_criterion(
            z1,
            z2,
            target
        )
    )


    logits1 = model.proxy_logits(
        z1
    )

    logits2 = model.proxy_logits(
        z2
    )


    proxy_loss1 = proxy_criterion(
        logits1,
        class1
    )

    proxy_loss2 = proxy_criterion(
        logits2,
        class2
    )


    proxy_loss = (
        0.5
        *
        (
            proxy_loss1
            +
            proxy_loss2
        )
    )


    total_loss = (
        contrastive_loss
        +
        lambda_proxy
        *
        proxy_loss
    )


    return {
        "total_loss":
            total_loss,

        "contrastive_loss":
            contrastive_loss,

        "proxy_loss":
            proxy_loss,

        "distances":
            distances,

        "logits1":
            logits1,

        "logits2":
            logits2
    }

In [15]:
# ============================================================
# SANITY CHECK
# ============================================================

batch = next(
    iter(train_pair_loader)
)


x1 = batch["x1"].to(DEVICE)

x2 = batch["x2"].to(DEVICE)

target = batch["target"].to(DEVICE)

class1 = batch["class1"].to(DEVICE)

class2 = batch["class2"].to(DEVICE)


model.eval()


with torch.no_grad():

    z1, z2 = model(
        x1,
        x2
    )


    result = compute_combined_loss(
        model=model,
        z1=z1,
        z2=z2,
        target=target,
        class1=class1,
        class2=class2,
        lambda_proxy=LAMBDA_PROXY
    )


print("=" * 72)
print("SANITY CHECK")
print("=" * 72)

print(
    "x1:",
    x1.shape
)

print(
    "z1:",
    z1.shape
)

print(
    "proxy logits:",
    result["logits1"].shape
)

print(
    "contrastive loss:",
    result["contrastive_loss"].item()
)

print(
    "proxy loss:",
    result["proxy_loss"].item()
)

print(
    "total loss:",
    result["total_loss"].item()
)

print(
    "embedding norm:",
    z1.norm(dim=1).mean().item()
)

print(
    "finite:",
    torch.isfinite(
        result["total_loss"]
    ).item()
)

SANITY CHECK
x1: torch.Size([64, 1, 64, 64])
z1: torch.Size([64, 128])
proxy logits: torch.Size([64, 12])
contrastive loss: 0.4918780028820038
proxy loss: 2.8017663955688477
total loss: 0.7720546722412109
embedding norm: 1.0
finite: True


In [16]:
# ============================================================
# EVALUATION — PAIRWISE + PROXY CLASSIFICATION
# ============================================================

@torch.no_grad()
def evaluate_model(
    model,
    loader,
    lambda_proxy
):

    model.eval()

    total_total_loss = 0.0
    total_contrastive_loss = 0.0
    total_proxy_loss = 0.0

    total_samples = 0

    all_targets = []
    all_distances = []

    all_class_labels = []
    all_class_predictions = []


    for batch in loader:

        x1 = batch["x1"].to(
            DEVICE,
            non_blocking=True
        )

        x2 = batch["x2"].to(
            DEVICE,
            non_blocking=True
        )

        target = batch["target"].to(
            DEVICE,
            non_blocking=True
        )

        class1 = batch["class1"].to(
            DEVICE,
            non_blocking=True
        )

        class2 = batch["class2"].to(
            DEVICE,
            non_blocking=True
        )


        with torch.autocast(
            device_type=DEVICE.type,
            dtype=(
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16
            ),
            enabled=AMP_ENABLED
        ):

            z1, z2 = model(
                x1,
                x2
            )


        result = compute_combined_loss(
            model=model,
            z1=z1,
            z2=z2,
            target=target,
            class1=class1,
            class2=class2,
            lambda_proxy=lambda_proxy
        )


        n = target.shape[0]


        total_total_loss += (
            result["total_loss"].item()
            * n
        )

        total_contrastive_loss += (
            result["contrastive_loss"].item()
            * n
        )

        total_proxy_loss += (
            result["proxy_loss"].item()
            * n
        )

        total_samples += n


        all_targets.append(
            target.cpu().numpy()
        )

        all_distances.append(
            result["distances"]
            .cpu()
            .numpy()
        )


        pred1 = (
            result["logits1"]
            .argmax(dim=1)
        )

        pred2 = (
            result["logits2"]
            .argmax(dim=1)
        )


        all_class_labels.append(
            torch.cat(
                [class1, class2]
            )
            .cpu()
            .numpy()
        )

        all_class_predictions.append(
            torch.cat(
                [pred1, pred2]
            )
            .cpu()
            .numpy()
        )


    # ========================================================
    # PAIRWISE METRICS
    # ========================================================

    targets = np.concatenate(
        all_targets
    )

    distances = np.concatenate(
        all_distances
    )


    positive = distances[
        targets == 1
    ]

    negative = distances[
        targets == 0
    ]


    d_pos = float(
        positive.mean()
    )

    d_neg = float(
        negative.mean()
    )

    gap = d_neg - d_pos


    pooled_variance = (
        0.5
        *
        (
            positive.var()
            +
            negative.var()
        )
    )


    d_prime = float(
        gap
        /
        np.sqrt(
            pooled_variance
            +
            1e-12
        )
    )


    pair_auc = float(
        roc_auc_score(
            targets,
            -distances
        )
    )


    # ========================================================
    # MULTICLASS PROXY METRICS
    # ========================================================

    class_labels = np.concatenate(
        all_class_labels
    )

    class_predictions = np.concatenate(
        all_class_predictions
    )


    proxy_accuracy = accuracy_score(
        class_labels,
        class_predictions
    )


    proxy_macro_f1 = f1_score(
        class_labels,
        class_predictions,
        average="macro",
        zero_division=0
    )


    proxy_balanced_accuracy = (
        balanced_accuracy_score(
            class_labels,
            class_predictions
        )
    )


    return {

        "total_loss":
            total_total_loss
            /
            total_samples,

        "contrastive_loss":
            total_contrastive_loss
            /
            total_samples,

        "proxy_loss":
            total_proxy_loss
            /
            total_samples,

        "pair_auc":
            pair_auc,

        "d_pos":
            d_pos,

        "d_neg":
            d_neg,

        "gap":
            gap,

        "d_prime":
            d_prime,

        "proxy_accuracy":
            proxy_accuracy,

        "proxy_macro_f1":
            proxy_macro_f1,

        "proxy_balanced_accuracy":
            proxy_balanced_accuracy
    }

In [17]:
# ============================================================
# TRAIN ONE EPOCH — CONTRASTIVE + PROXY
# ============================================================

def train_one_epoch(
    model,
    loader,
    dataset,
    optimizer,
    scaler,
    epoch,
    lambda_proxy
):

    model.train()

    dataset.set_epoch(
        epoch
    )


    total_total_loss = 0.0
    total_contrastive_loss = 0.0
    total_proxy_loss = 0.0

    total_samples = 0

    all_targets = []
    all_distances = []

    all_class_labels = []
    all_class_predictions = []


    start_time = time.perf_counter()


    for batch in loader:

        x1 = batch["x1"].to(
            DEVICE,
            non_blocking=True
        )

        x2 = batch["x2"].to(
            DEVICE,
            non_blocking=True
        )

        target = batch["target"].to(
            DEVICE,
            non_blocking=True
        )

        class1 = batch["class1"].to(
            DEVICE,
            non_blocking=True
        )

        class2 = batch["class2"].to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(
            device_type=DEVICE.type,
            dtype=(
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16
            ),
            enabled=AMP_ENABLED
        ):

            z1, z2 = model(
                x1,
                x2
            )


        result = compute_combined_loss(
            model=model,
            z1=z1,
            z2=z2,
            target=target,
            class1=class1,
            class2=class2,
            lambda_proxy=lambda_proxy
        )


        loss = result[
            "total_loss"
        ]


        if AMP_ENABLED:

            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            loss.backward()

            optimizer.step()


        n = target.shape[0]


        total_total_loss += (
            result["total_loss"]
            .detach()
            .item()
            * n
        )

        total_contrastive_loss += (
            result["contrastive_loss"]
            .detach()
            .item()
            * n
        )

        total_proxy_loss += (
            result["proxy_loss"]
            .detach()
            .item()
            * n
        )

        total_samples += n


        all_targets.append(
            target.detach()
            .cpu()
            .numpy()
        )

        all_distances.append(
            result["distances"]
            .detach()
            .cpu()
            .numpy()
        )


        pred1 = (
            result["logits1"]
            .detach()
            .argmax(dim=1)
        )

        pred2 = (
            result["logits2"]
            .detach()
            .argmax(dim=1)
        )


        all_class_labels.append(
            torch.cat(
                [class1, class2]
            )
            .detach()
            .cpu()
            .numpy()
        )

        all_class_predictions.append(
            torch.cat(
                [pred1, pred2]
            )
            .detach()
            .cpu()
            .numpy()
        )


    targets = np.concatenate(
        all_targets
    )

    distances = np.concatenate(
        all_distances
    )


    positive = distances[
        targets == 1
    ]

    negative = distances[
        targets == 0
    ]


    class_labels = np.concatenate(
        all_class_labels
    )

    class_predictions = np.concatenate(
        all_class_predictions
    )


    return {

        "total_loss":
            total_total_loss
            /
            total_samples,

        "contrastive_loss":
            total_contrastive_loss
            /
            total_samples,

        "proxy_loss":
            total_proxy_loss
            /
            total_samples,

        "pair_auc":
            float(
                roc_auc_score(
                    targets,
                    -distances
                )
            ),

        "d_pos":
            float(
                positive.mean()
            ),

        "d_neg":
            float(
                negative.mean()
            ),

        "gap":
            float(
                negative.mean()
                -
                positive.mean()
            ),

        "proxy_accuracy":
            accuracy_score(
                class_labels,
                class_predictions
            ),

        "proxy_macro_f1":
            f1_score(
                class_labels,
                class_predictions,
                average="macro",
                zero_division=0
            ),

        "seconds":
            (
                time.perf_counter()
                -
                start_time
            )
    }

In [18]:
# ============================================================
# SMOKE TEST — LAMBDA_PROXY = 0.10
# ============================================================

SMOKE_EPOCHS = 3


set_seed(
    RANDOM_STATE
)


model = SiameseWithProxies(
    n_classes=N_CLASSES,
    embedding_dim=EMBEDDING_DIM,
    temperature=PROXY_TEMPERATURE
).to(DEVICE)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


initial_state = copy.deepcopy(
    model.state_dict()
)


print("=" * 132)
print(
    f"SIAMESE CONTRASTIVE + PROXY — lambda={LAMBDA_PROXY}"
)
print("=" * 132)


for epoch in range(
    1,
    SMOKE_EPOCHS + 1
):

    train_metrics = train_one_epoch(
        model=model,
        loader=train_pair_loader,
        dataset=train_pair_dataset,
        optimizer=optimizer,
        scaler=scaler,
        epoch=epoch,
        lambda_proxy=LAMBDA_PROXY
    )


    val_metrics = evaluate_model(
        model=model,
        loader=val_pair_loader,
        lambda_proxy=LAMBDA_PROXY
    )


    print(
        f"Epoch {epoch:02d}/{SMOKE_EPOCHS}"

        f" | total "
        f"{train_metrics['total_loss']:.4f}"

        f" | contrast "
        f"{train_metrics['contrastive_loss']:.4f}"

        f" | proxy "
        f"{train_metrics['proxy_loss']:.4f}"

        f" | train AUC "
        f"{train_metrics['pair_auc']:.4f}"

        f" | val AUC "
        f"{val_metrics['pair_auc']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.4f}"

        f" | proxy Acc "
        f"{val_metrics['proxy_accuracy']:.4f}"

        f" | proxy F1 "
        f"{val_metrics['proxy_macro_f1']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"
    )

SIAMESE CONTRASTIVE + PROXY — lambda=0.1
Epoch 01/3 | total 0.5781 | contrast 0.3706 | proxy 2.0748 | train AUC 0.6408 | val AUC 0.6363 | gap 0.1170 | d' 0.4959 | proxy Acc 0.2683 | proxy F1 0.2254 | 38.1s
Epoch 02/3 | total 0.5587 | contrast 0.3618 | proxy 1.9696 | train AUC 0.6577 | val AUC 0.6555 | gap 0.1254 | d' 0.5645 | proxy Acc 0.2983 | proxy F1 0.2468 | 20.6s
Epoch 03/3 | total 0.5523 | contrast 0.3580 | proxy 1.9425 | train AUC 0.6664 | val AUC 0.6553 | gap 0.1105 | d' 0.5647 | proxy Acc 0.3088 | proxy F1 0.2735 | 45.8s


In [19]:
# ============================================================
# EXTENDED SMOKE TEST — EPOCHS 4-6
# ============================================================

EXTRA_EPOCHS = 3

print("=" * 136)
print(
    f"SIAMESE CONTRASTIVE + PROXY — EXTENDED — lambda={LAMBDA_PROXY}"
)
print("=" * 136)


for epoch in range(
    SMOKE_EPOCHS + 1,
    SMOKE_EPOCHS + EXTRA_EPOCHS + 1
):

    train_metrics = train_one_epoch(
        model=model,
        loader=train_pair_loader,
        dataset=train_pair_dataset,
        optimizer=optimizer,
        scaler=scaler,
        epoch=epoch,
        lambda_proxy=LAMBDA_PROXY
    )


    val_metrics = evaluate_model(
        model=model,
        loader=val_pair_loader,
        lambda_proxy=LAMBDA_PROXY
    )


    print(
        f"Epoch {epoch:02d}/6"

        f" | total "
        f"{train_metrics['total_loss']:.4f}"

        f" | contrast "
        f"{train_metrics['contrastive_loss']:.4f}"

        f" | proxy "
        f"{train_metrics['proxy_loss']:.4f}"

        f" | train AUC "
        f"{train_metrics['pair_auc']:.4f}"

        f" | val AUC "
        f"{val_metrics['pair_auc']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.4f}"

        f" | proxy Acc "
        f"{val_metrics['proxy_accuracy']:.4f}"

        f" | proxy F1 "
        f"{val_metrics['proxy_macro_f1']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"
    )

SIAMESE CONTRASTIVE + PROXY — EXTENDED — lambda=0.1
Epoch 04/6 | total 0.5480 | contrast 0.3572 | proxy 1.9082 | train AUC 0.6686 | val AUC 0.6476 | gap 0.0998 | d' 0.5357 | proxy Acc 0.2961 | proxy F1 0.2764 | 21.4s
Epoch 05/6 | total 0.5416 | contrast 0.3524 | proxy 1.8924 | train AUC 0.6789 | val AUC 0.6631 | gap 0.1167 | d' 0.5909 | proxy Acc 0.3170 | proxy F1 0.2884 | 21.0s
Epoch 06/6 | total 0.5380 | contrast 0.3508 | proxy 1.8721 | train AUC 0.6825 | val AUC 0.6571 | gap 0.1008 | d' 0.5721 | proxy Acc 0.2997 | proxy F1 0.2845 | 20.7s


In [20]:
# ============================================================
# SINGLE FCGR DATASET
# ============================================================

class SingleFCGRDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = fcgr_memmap

        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )

        self.labels = (
            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )


    def __len__(self):

        return len(self.metadata)


    def __getitem__(
        self,
        index
    ):

        x = np.array(
            self.fcgr_memmap[
                int(self.rows[index])
            ],
            dtype=np.float32,
            copy=True
        )

        return {
            "x":
                torch.from_numpy(x)
                .unsqueeze(0),

            "label":
                torch.tensor(
                    self.labels[index],
                    dtype=torch.long
                )
        }

In [21]:
# ============================================================
# SINGLE SAMPLE LOADERS
# ============================================================

reference_dataset = SingleFCGRDataset(
    metadata=train_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row
)


validation_dataset = SingleFCGRDataset(
    metadata=val_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row
)


reference_loader = DataLoader(
    reference_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=AMP_ENABLED
)


validation_loader = DataLoader(
    validation_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=AMP_ENABLED
)


print(
    "Reference samples:",
    len(reference_dataset)
)

print(
    "Validation samples:",
    len(validation_dataset)
)

Reference samples: 25332
Validation samples: 9753


In [22]:
# ============================================================
# EXTRACT EMBEDDINGS
# ============================================================

@torch.no_grad()
def extract_embeddings(
    encoder,
    loader
):

    encoder.eval()

    all_embeddings = []
    all_labels = []


    for batch in loader:

        x = batch["x"].to(
            DEVICE,
            non_blocking=True
        )


        with torch.autocast(
            device_type=DEVICE.type,
            dtype=(
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16
            ),
            enabled=AMP_ENABLED
        ):

            z = encoder(x)


        all_embeddings.append(
            z.float()
            .cpu()
            .numpy()
        )

        all_labels.append(
            batch["label"]
            .numpy()
        )


    return (
        np.concatenate(
            all_embeddings,
            axis=0
        ),

        np.concatenate(
            all_labels,
            axis=0
        )
    )


reference_embeddings, reference_labels = (
    extract_embeddings(
        model.encoder,
        reference_loader
    )
)


val_embeddings, val_labels = (
    extract_embeddings(
        model.encoder,
        validation_loader
    )
)


print(
    "Reference:",
    reference_embeddings.shape
)

print(
    "Validation:",
    val_embeddings.shape
)

print(
    "Mean norm:",
    np.linalg.norm(
        val_embeddings,
        axis=1
    ).mean()
)

Reference: (25332, 128)
Validation: (9753, 128)
Mean norm: 1.0


In [23]:
# ============================================================
# UNIQUE VALIDATION — LEARNED PROXY CLASSIFICATION
# ============================================================

model.eval()


with torch.no_grad():

    normalized_proxies = (
        F.normalize(
            model.proxies,
            p=2,
            dim=1
        )
        .float()
        .cpu()
        .numpy()
    )


proxy_scores = (
    val_embeddings
    @
    normalized_proxies.T
)


proxy_predictions = (
    proxy_scores.argmax(
        axis=1
    )
)


unique_proxy_accuracy = accuracy_score(
    val_labels,
    proxy_predictions
)


unique_proxy_macro_f1 = f1_score(
    val_labels,
    proxy_predictions,
    average="macro",
    zero_division=0
)


unique_proxy_balanced = balanced_accuracy_score(
    val_labels,
    proxy_predictions
)


print("=" * 72)
print("UNIQUE VALIDATION — LEARNED PROXIES")
print("=" * 72)

print(
    "Accuracy:",
    f"{unique_proxy_accuracy:.6f}"
)

print(
    "Macro-F1:",
    f"{unique_proxy_macro_f1:.6f}"
)

print(
    "Balanced Accuracy:",
    f"{unique_proxy_balanced:.6f}"
)

UNIQUE VALIDATION — LEARNED PROXIES
Accuracy: 0.293038
Macro-F1: 0.275048
Balanced Accuracy: 0.305880


In [24]:
# ============================================================
# PROTOTYPE CLASSIFICATION
# ============================================================

prototypes = []


for class_id in range(
    N_CLASSES
):

    class_embeddings = (
        reference_embeddings[
            reference_labels
            ==
            class_id
        ]
    )


    prototype = (
        class_embeddings.mean(
            axis=0
        )
    )


    prototype = (
        prototype
        /
        (
            np.linalg.norm(prototype)
            +
            1e-12
        )
    )


    prototypes.append(
        prototype
    )


prototypes = np.stack(
    prototypes
)


prototype_distances = np.linalg.norm(

    val_embeddings[:, None, :]
    -
    prototypes[None, :, :],

    axis=2
)


prototype_predictions = (
    prototype_distances.argmin(
        axis=1
    )
)


print("=" * 72)
print("UNIQUE VALIDATION — PROTOTYPES")
print("=" * 72)

print(
    "Accuracy:",
    f"{accuracy_score(val_labels, prototype_predictions):.6f}"
)

print(
    "Macro-F1:",
    f"{f1_score(val_labels, prototype_predictions, average='macro', zero_division=0):.6f}"
)

print(
    "Balanced Accuracy:",
    f"{balanced_accuracy_score(val_labels, prototype_predictions):.6f}"
)

UNIQUE VALIDATION — PROTOTYPES
Accuracy: 0.289039
Macro-F1: 0.259636
Balanced Accuracy: 0.299722
